# Run All Augmentation Notebooks

Batch runner for all `augment_*.ipynb` notebooks in this folder.


In [ ]:
from pathlib import Path
import json
import os
import re
import sys
import time
from contextlib import contextmanager

import nbformat
import pandas as pd
from nbclient import NotebookClient
from tqdm.auto import tqdm

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'pyproject.toml').exists() else next(
    path for path in [CWD, *CWD.parents] if (path / 'pyproject.toml').exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_augmentation.common import resolve_augmentation_artifact_dir
from feature_selection.common import machine_name_from_env, resolve_feature_source, variant_name, variants_to_run


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


@contextmanager
def pushd(target_dir: Path):
    prev = Path.cwd()
    os.chdir(target_dir)
    try:
        yield
    finally:
        os.chdir(prev)


@contextmanager
def temp_environ(updates: dict[str, str]):
    previous = {key: os.environ.get(key) for key in updates}
    for key, value in updates.items():
        os.environ[key] = str(value)
    try:
        yield
    finally:
        for key, value in previous.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = value


REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_DIR = REPO_ROOT / 'feature_augmentation' / 'mean_std_oversampling'
RUNNER_NAME = 'run_all_augmentation.ipynb'
EXECUTE_TIMEOUT = None
KERNEL_NAME = 'python3'
CONTINUE_ON_ERROR = True
SKIP_EXISTING = True
USE_VARIANT_ARTIFACT_DIRS = True
MACHINE_NAME = machine_name_from_env()
VARIANTS = variants_to_run(default_both=True)
SKIP_NOTEBOOKS = {'visualize_augmentation_results.ipynb'}

targets = sorted(
    p for p in NOTEBOOK_DIR.glob('augment_*.ipynb')
    if p.name not in {RUNNER_NAME, *SKIP_NOTEBOOKS}
)

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Runner: {RUNNER_NAME}')
print(f'Machine: {MACHINE_NAME}')
print('Variants:', [variant_name(include_xxx) for include_xxx in VARIANTS])
print('Skipped notebooks:', sorted(SKIP_NOTEBOOKS))
print(f'Targets: {len(targets)}')
for p in targets:
    print(f' - {p.name}')

DATASET_KEY_PATTERN = re.compile(r"^DATASET_KEY\s*=\s*['\"]([^'\"]+)['\"]\s*$", re.MULTILINE)


def extract_dataset_key(notebook_path: Path) -> str | None:
    notebook = nbformat.read(notebook_path, as_version=4)
    for cell in notebook.cells:
        if cell.get('cell_type') != 'code':
            continue
        match = DATASET_KEY_PATTERN.search(cell.get('source', ''))
        if match:
            return match.group(1)
    return None


NOTEBOOK_DATASET_KEYS = {path: extract_dataset_key(path) for path in targets}

MISSING_SOURCE_NOTEBOOKS = {
    path.name
    for path, dataset_key in NOTEBOOK_DATASET_KEYS.items()
    if dataset_key and not resolve_feature_source(REPO_ROOT, dataset_key).exists()
}
if MISSING_SOURCE_NOTEBOOKS:
    print('Skipping notebooks with missing source CSVs:', sorted(MISSING_SOURCE_NOTEBOOKS))
    targets = [path for path in targets if path.name not in MISSING_SOURCE_NOTEBOOKS]
    NOTEBOOK_DATASET_KEYS = {path: NOTEBOOK_DATASET_KEYS[path] for path in targets}


def resolve_existing_meta_path(notebook_path: Path, *, include_xxx: bool) -> Path | None:
    dataset_key = NOTEBOOK_DATASET_KEYS.get(notebook_path)
    if not dataset_key:
        return None
    artifact_dir = resolve_augmentation_artifact_dir(
        REPO_ROOT,
        include_xxx=include_xxx,
        use_variant_dirs=USE_VARIANT_ARTIFACT_DIRS,
    )
    return artifact_dir / f'{dataset_key}_run_metadata.json'


print(f'Skip existing artifact runs: {SKIP_EXISTING}')


In [ ]:
results: list[dict[str, object]] = []

for include_xxx in VARIANTS:
    current_variant = variant_name(include_xxx)

    for notebook_path in tqdm(
        targets,
        total=len(targets),
        desc=f'Running notebooks | {current_variant}',
    ):
        row = {
            'variant': current_variant,
            'notebook': notebook_path.name,
            'dataset_key': NOTEBOOK_DATASET_KEYS.get(notebook_path),
            'status': 'pending',
            'seconds': None,
            'error': None,
        }
        started = time.perf_counter()
        existing_meta_path = resolve_existing_meta_path(notebook_path, include_xxx=include_xxx)
        row['meta_path'] = str(existing_meta_path) if existing_meta_path is not None else None

        if SKIP_EXISTING and existing_meta_path is not None and existing_meta_path.exists():
            row['status'] = 'skipped_existing'
            row['seconds'] = 0.0
            results.append(row)
            continue

        try:
            nb = nbformat.read(notebook_path, as_version=4)
            with pushd(NOTEBOOK_DIR), temp_environ({
                'SER_MACHINE': MACHINE_NAME,
                'SER_INCLUDE_XXX': 'true' if include_xxx else 'false',
                'SER_RUN_BOTH_XXX_VARIANTS': '0',
                'SER_USE_VARIANT_ARTIFACT_DIRS': '1',
            }):
                client = NotebookClient(
                    nb,
                    timeout=EXECUTE_TIMEOUT,
                    kernel_name=KERNEL_NAME,
                )
                client.execute()
            row['status'] = 'ok'
        except Exception as exc:  # noqa: BLE001
            row['status'] = 'failed'
            row['error'] = f'{type(exc).__name__}: {exc}'
            if not CONTINUE_ON_ERROR:
                row['seconds'] = round(time.perf_counter() - started, 3)
                results.append(row)
                raise
        finally:
            if row['seconds'] is None:
                row['seconds'] = round(time.perf_counter() - started, 3)
            results.append(row)

results_df = pd.DataFrame(results)
display(results_df)

if not results_df.empty:
    print('\nRun status counts:')
    print(results_df['status'].value_counts())
